In [0]:
df = spark.table("bankingdata.bronze.product")

In [0]:
df.columns

### Handling rescue data column

In [0]:
from pyspark.sql.functions import *

In [0]:
df_rescued = df.filter(col("_rescued_data").isNotNull())
df = df.filter(col("_rescued_data").isNull())

In [0]:
df_rescued_final = df_rescued.select(
    col("ProductID"),          
    col("_rescued_data"),       
    col("source_file"),        
    col("ingestion_time")       
)

In [0]:
df_rescued_final.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bankingdata.rescueddata.product_rescued")

### Schema

In [0]:
df = df.select(
    col("ProductID").cast("int").alias("ProductID"),
    col("ProductName").cast("string").alias("ProductName"),
    col("ProductType").cast("string").alias("ProductType"),
    col("InterestRate").cast("double").alias("InterestRate"),
    col("ModifiedDate").cast("timestamp").alias("ModifiedDate")
)

### Primary Key Validation

In [0]:
df = df.filter(
    col("ProductID").isNotNull() &
    (trim(col("ProductID").cast("string")) != "") &
    (col("ProductID") > 0)
)

### Business Quality Check


In [0]:
df = df.filter(
    col("ProductName").isNotNull() &
    (length(trim(col("ProductName"))) > 2) &

    col("ProductType").isNotNull() &
    col("ProductType").isin("Loan", "Deposit") &

    col("InterestRate").isNotNull() &
    (col("InterestRate") >= 0) &
    (col("InterestRate") <= 100) &

    col("ModifiedDate").isNotNull() &
    (col("ModifiedDate") <= current_timestamp())
)

### Remove Duplicates

In [0]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("ProductID") \
                    .orderBy(col("ModifiedDate").desc())

df = df.withColumn("rn", row_number().over(window_spec)) \
                   .filter(col("rn") == 1) \
                   .drop("rn")

### Delta table

In [0]:
# save table
silver_table = "bankingdata.silver.product"

df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(silver_table)

print("Silver table created:", silver_table)